# graph

> Graph-spine construction, commit, and skeptical-lens verification. Builds Document +
> Segment nodes and STARTS_WITH / NEXT / PART_OF edges (pure), commits them through the
> graph-storage capability via the job queue, and re-queries the graph to verify.

In [ ]:
#| default_exp graph

In [ ]:
#| export
from uuid import uuid4
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

from cjm_plugin_system.core.queue import JobQueue, JobStatus
from cjm_context_graph_primitives.locators import FileRef
from cjm_context_graph_primitives.provenance import SourceRef
from cjm_context_graph_primitives.graph import GraphContext, GraphNode
# Stage 4: typed query expressions + results (importing the result classes IS
# the host-side wire registration, F8)
from cjm_context_graph_primitives.query import (
    NodeQuery, EdgeQuery, PropertyPredicate, RelationPredicate, OrderBy,
    NodeQueryResult, EdgeQueryResult,
)
from cjm_graph_domains.domains.structure import Document, Segment
from cjm_graph_domains.domains.relations import StructureRelations

from cjm_transcript_decomp_core.models import DecompSegment

In [ ]:
#| export
def build_source_ref(
    seg: DecompSegment,  # Aligned segment carrying upstream provenance
    manifest_path: str,  # Path to the consumed transcription run manifest
) -> Optional[SourceRef]:  # Provenance ref, or None when source info is absent
    """Build a manifest-anchored SourceRef for one segment (CR-19 shape).

    Identity = content_hash over the consumed segment text — verifiable
    regardless of locator resolution, so the E13/D3 dangling-row problem is now
    fixed structurally rather than worked around. Locator = FileRef to the
    consumed run manifest (the file this core actually reads; no more
    `external:<path>` field abuse). slice=None: the consumed region sits inside
    ONE MANIFEST ROW's text, which a slice framed to the manifest FILE cannot
    honestly express (the D4 coordinate-space friction, carried until stage 5
    gives the intra-graph GraphNodeRef(Transcript) + CharSlice anchor). The char
    coordinates and upstream job id stay queryable as Segment node properties.
    """
    if not seg.source_job_id:
        return None
    return SourceRef(
        locator=FileRef(path=manifest_path),
        content_hash=SourceRef.compute_hash(seg.text.encode()),
    )

In [ ]:
#| export
def build_graph_payload(
    title: str,                     # Document title
    segments: List[DecompSegment],  # Ordered aligned segments (source-coord timing)
    manifest_path: str,             # Consumed manifest path (for SourceRefs)
    media_type: str = "audio",      # Document media type
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], str, List[str]]:  # (nodes, edges, doc_id, seg_ids)
    """Build the (nodes, edges, document_id, segment_ids) graph payload.

    Pure: assembles Document + Segment nodes and STARTS_WITH / NEXT / PART_OF
    edges as wire dicts; no capability calls (commit happens in `commit_graph`).
    """
    doc = Document(id=str(uuid4()), title=title, media_type=media_type)
    doc_node = doc.to_graph_node(sources=[])

    segment_nodes = []
    for seg in segments:
        ref = build_source_ref(seg, manifest_path)
        node = Segment(
            id=str(uuid4()), text=seg.text, index=seg.index,
            start_time=seg.start_time, end_time=seg.end_time,
            start_char=seg.start_char, end_char=seg.end_char,
        ).to_graph_node(sources=[ref] if ref else [])
        if seg.source_job_id:
            node.properties["source_job_id"] = seg.source_job_id
        segment_nodes.append(node)

    edges: List[Dict[str, Any]] = []
    if segment_nodes:
        edges.append({"id": str(uuid4()), "source_id": doc_node.id,
                      "target_id": segment_nodes[0].id,
                      "relation_type": StructureRelations.STARTS_WITH, "properties": {}})
    for i, node in enumerate(segment_nodes):
        edges.append({"id": str(uuid4()), "source_id": node.id, "target_id": doc_node.id,
                      "relation_type": StructureRelations.PART_OF, "properties": {}})
        if i < len(segment_nodes) - 1:
            edges.append({"id": str(uuid4()), "source_id": node.id,
                          "target_id": segment_nodes[i + 1].id,
                          "relation_type": StructureRelations.NEXT, "properties": {}})

    nodes = [doc_node.to_dict()] + [n.to_dict() for n in segment_nodes]
    return nodes, edges, doc_node.id, [n.id for n in segment_nodes]

In [ ]:
#| export
GRAPH_TASK = "graph-storage"  # The graph-storage adapter task (explicit task channel, stage 4)


async def graph_task(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability instance id
    method: str,      # Adapter method (e.g. "query_nodes", "add_nodes")
    **kwargs,         # Typed-method kwargs (wire dicts ok; the in-worker adapter normalizes)
) -> Any:  # Typed task result (wire-decoded host-side)
    """Invoke a graph-storage adapter method through the queue's task channel (stage 4)."""
    jid = await queue.submit(graph_id, task=GRAPH_TASK, method=method, **kwargs)
    job = await queue.wait_for_job(jid)
    if job.status != JobStatus.completed:
        raise RuntimeError(f"{graph_id} {method} {job.status}: {job.error}")
    return job.result


In [ ]:
#| export
async def commit_graph(
    queue: JobQueue,              # Started job queue
    graph_id: str,                # Graph-storage capability id
    nodes: List[Dict[str, Any]],  # Node wire dicts from build_graph_payload
    edges: List[Dict[str, Any]],  # Edge wire dicts from build_graph_payload
) -> Dict[str, int]:  # {"nodes": n, "edges": m} created counts
    """Commit nodes then edges to the graph capability via the task channel.

    Goes through `JobQueue` (not a direct `execute_plugin_async`) so graph writes
    get empirical samples + sysmon attribution like every other capability call
    (the GUI GraphService used the direct path — pass-2 two-invocation-paths note).
    Stage 4: the explicit task channel replaces `action=` dispatch; the adapter's
    typed add_nodes/add_edges return created-id lists.
    """
    node_ids = await graph_task(queue, graph_id, "add_nodes", nodes=nodes)
    edge_ids = await graph_task(queue, graph_id, "add_edges", edges=edges)
    return {"nodes": len(node_ids or []), "edges": len(edge_ids or [])}

In [ ]:
#| export
@dataclass
class VerificationResult:
    """Skeptical-lens verification computed by querying the graph directly."""
    document_id: str       # Verified Document node id
    title: str             # Document title (read back from the graph)
    segment_count: int     # Segment nodes found
    has_starts_with: bool  # >=1 STARTS_WITH edge from the Document
    next_chain_complete: bool  # NEXT edge count == segment_count - 1
    part_of_complete: bool     # PART_OF edge count == segment_count
    all_have_timing: bool      # Every segment has start_time + end_time
    all_have_sources: bool     # Every segment has >=1 SourceRef
    source_locators: List[str] = field(default_factory=list)  # Distinct source locator URIs

    @property
    def ok(self) -> bool:  # True when every structural check passes
        """All structural checks pass."""
        return (self.has_starts_with and self.next_chain_complete
                and self.part_of_complete and self.all_have_timing
                and self.all_have_sources)

In [ ]:
#| export
async def verify_document(
    queue: JobQueue,   # Started job queue
    graph_id: str,     # Graph-storage capability id
    document_id: str,  # Document node id to verify
) -> Optional[VerificationResult]:  # Result, or None if the document is not found
    """Verify a committed document via server-side AGGREGATES (stage 4; D13).

    Skeptical lens preserved: every check still reads STORAGE, not run state —
    but as typed count queries + one bounded sources-only projection instead of
    materializing the whole depth-2 neighborhood (~3,580 nodes / ~7,260 edges at
    Supernova scale) into the consumer. The NEXT-chain count uses the promoted
    `EdgeQuery.source_related` endpoint constraint ("NEXT edges whose source is
    PART_OF this document").
    """
    doc = await graph_task(queue, graph_id, "get_node", node_id=document_id)
    if doc is None:
        return None
    doc_props = doc.properties if isinstance(doc, GraphNode) else ((doc or {}).get("properties") or {})
    part_of = RelationPredicate("PART_OF", node_id=document_id)

    async def _ncount(**kw):
        q = NodeQuery(label="Segment", related=part_of, count=True, **kw)
        res = await graph_task(queue, graph_id, "query_nodes", query=q.to_dict())
        return int(res.count or 0)

    async def _ecount(**kw):
        q = EdgeQuery(count=True, **kw)
        res = await graph_task(queue, graph_id, "query_edges", query=q.to_dict())
        return int(res.count or 0)

    n = await _ncount()
    starts_with = await _ecount(relation_type=StructureRelations.STARTS_WITH,
                                source_id=document_id)
    next_count = await _ecount(relation_type=StructureRelations.NEXT,
                               source_related=part_of)
    part_of_count = await _ecount(relation_type=StructureRelations.PART_OF,
                                  target_id=document_id)
    missing_start = await _ncount(where=[PropertyPredicate("start_time", "is_null")])
    missing_end = await _ncount(where=[PropertyPredicate("end_time", "is_null")])

    # Distinct locators + missing-sources in ONE bounded pass: sources is a
    # COLUMN (not a property), so an emptiness predicate isn't expressible yet —
    # recorded promotion candidate; the projection rows carry id + sources only.
    sq = NodeQuery(label="Segment", related=part_of, project=["sources"])
    res = await graph_task(queue, graph_id, "query_nodes", query=sq.to_dict())
    missing_sources = 0
    locators = set()
    for r in (res.rows or []):
        sources = r.get("sources") or []
        if not sources:
            missing_sources += 1
        for src in sources:
            ref = SourceRef.from_dict(src) if isinstance(src, dict) else src
            locators.add(ref.locator.to_uri())

    return VerificationResult(
        document_id=document_id,
        title=doc_props.get("title", "Untitled"),
        segment_count=n,
        has_starts_with=starts_with >= 1,
        next_chain_complete=next_count == max(0, n - 1),
        part_of_complete=part_of_count == n,
        all_have_timing=(missing_start == 0 and missing_end == 0),
        all_have_sources=missing_sources == 0,
        source_locators=sorted(locators),
    )

In [ ]:
# graph payload smoke checks (pure; no capabilities)
_segs = [
    DecompSegment(index=0, text="Alpha.", start_time=0.0, end_time=2.0,
                  start_char=0, end_char=6, source_job_id="j0", source_provider_id="p"),
    DecompSegment(index=1, text="Beta.", start_time=2.0, end_time=4.0,
                  start_char=0, end_char=5, source_job_id="j1", source_provider_id="p"),
    DecompSegment(index=2, text="Gamma.", start_time=4.0, end_time=6.0,
                  start_char=0, end_char=6, source_job_id="j2", source_provider_id="p"),
]
nodes, edges, doc_id, seg_ids = build_graph_payload("Test Doc", _segs, "/tmp/m.json")
assert len(nodes) == 4  # 1 Document + 3 Segments
assert len(seg_ids) == 3
_rels = [e["relation_type"] for e in edges]
assert _rels.count("STARTS_WITH") == 1
assert _rels.count("NEXT") == 2
assert _rels.count("PART_OF") == 3

_sr = build_source_ref(_segs[0], "/tmp/m.json")
assert _sr.locator.to_uri() == "file:/tmp/m.json"
assert _sr.slice is None
assert _sr.verify(b"Alpha.")  # identity holds regardless of locator resolution
# upstream correlation id + char coords ride as node properties
assert nodes[1]["properties"]["source_job_id"] == "j0"
assert nodes[1]["properties"]["start_char"] == 0
assert build_source_ref(DecompSegment(0, "x", 0.0, 1.0, None, None, None, None), "/tmp/m.json") is None
print("graph payload checks OK")

graph payload checks OK
